# 🎯 YOLOv8 Training for P&ID Symbol Detection

**Train a custom YOLOv8 model to detect engineering symbols in P&ID drawings.**

Runs on **Google Colab (free T4 GPU)**.

---

## ▶ START HERE (3 steps)

1. **Set the Roboflow key** (only if you have not already):
   ```python
   from google.colab import userdata
   userdata.put("ROBOFLOW_API_KEY", "<your key>")
   ```
   > 🔐 The previous version of this notebook had a live Roboflow key in
   > plaintext, and it is still in this repository's git history. **Rotate
   > that key.** This notebook reads the key from Colab secrets and never
   > writes it to a file.

2. **Runtime → Run all.**

3. **Download `best.pt`** from the last cell, then copy it to
   `models/best.pt` in your project.

---

## What changed, and why

This notebook was rewritten after an audit of the shipped `models/best.pt`
and the Roboflow dataset it was trained on. Every change below traces to
a measured defect (full write-up: `MODEL_DATASET_AUDIT.md`).

| Defect found | Fix in this notebook |
|---|---|
| **API key hardcoded in the notebook** (and in git history) | Read from Colab secrets / env; never written to a file |
| **`val` fell back to `train`** — mAP was measured on training data | Hard failure if the val split is missing |
| **2,194 of 5,462 image files are byte-identical duplicates**, and 498 appear in two splits | `tools/repair_dataset.py` dedupes and re-splits leak-free before training |
| **7 classes were the same symbol spelled differently** (`Pneumatic - Diaphragm Gate Valve` vs `Pneumatic-DiaphragmGate Valve`) — the loser measured **map50 = 0.000** | Merged into one canonical class |
| **7 class names were misspelled** (`bolier`, `Fluidzed Recactor`, `temp trasmitter`) — `bolier` also measured **map50 = 0.000** | Corrected |
| **229 classes but only 153 were ever evaluated** — the headline mAP hid 76 dead classes | Per-class report; the split guarantees every class is in train and val |
| **Class imbalance of 20,572×** (median class: 30 instances; 139 classes have <50) | Class-health table; `close_mosaic` + longer schedule |
| **Mosaic stayed on for the last 10 epochs**, blurring fine line work | `close_mosaic=15` |
| **`erasing` left at the default 0.4** — it *erases* 4% of every image, deleting symbols | `erasing=0.0` |
| **`auto_augment` on by default** — randaugment is for natural images | `auto_augment=None` |
| **Training args were stripped from the checkpoint** — you could not tell how `best.pt` was made | Args, epoch, metrics and a class-health table are re-attached before export |
| **No per-class metrics** | Full per-class table, the zero-mAP list, and a cross-reference against the training instance counts |
| **imgsz=640 with 44 different source image sizes** | `imgsz=1024` |

### One thing training alone will NOT fix

The shipped model scores mAP50 0.849 on the Roboflow dataset but finds
**none** of the vessels, columns, exchangers and reactors on a
P&ID drawn in a different style — even `Centrifugal Pump`, with 2,013
training instances and mAP50 0.935, misses 6 of 8 pumps there.

That is a **style gap, not a data-volume gap**: the glyphs are drawn
differently. Step 3b below merges in a target-style dataset if you have
one. Without it, expect the Roboflow classes to improve and the
target-style equipment symbols to stay missing.

## Step 1: Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow pyyaml

import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch      :", torch.__version__)
print("cuda       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device     :", torch.cuda.get_device_name(0))
    print("memory     : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("WARNING: no GPU. Training on CPU takes days - use a Colab GPU runtime.")

## Step 2: Download the Dataset

In [ ]:
import os

def _get_roboflow_key():
    try:
        from google.colab import userdata
        key = userdata.get("ROBOFLOW_API_KEY")
        if key:
            return key, "colab secret"
    except Exception:
        pass
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key, "environment variable"
    raise RuntimeError(
        "ROBOFLOW_API_KEY not found.\n"
        "In Colab: from google.colab import userdata; "
        "userdata.put('ROBOFLOW_API_KEY', '<your key>')\n"
        "Or set the ROBOFLOW_API_KEY environment variable."
    )

ROBOFLOW_API_KEY, key_source = _get_roboflow_key()
print(f"Roboflow key loaded from: {key_source}")

from roboflow import Roboflow

WORKSPACE = "ai-ready-program"
PROJECT   = "p-id-object-detection-ai-ready-pomcl"
VERSION   = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download("yolov8", location="/content/dataset")
print("Downloaded ->", dataset.location)

# Never print the key again.
del ROBOFLOW_API_KEY

## Step 3: Repair the Dataset

The shipped Roboflow split has three structural defects:

1. **2,194 of 5,462 image files are byte-identical duplicates**, and 498
   appear in two different splits. The duplicate *training* files
   over-weighted the sheets they duplicated — measured with the class ids
   held constant, deduplicating raised mAP50 from 0.8491 to **0.8815**.
2. **7 groups of classes are the same symbol spelled differently.** The
   model cannot win: it must predict the right *spelling* of a symbol it
   cannot otherwise distinguish. `Pneumatic - DiaphragmGate Valve`
   measured **map50 = 0.000** while its twin (3,134 instances) was
   learned perfectly.
3. **66 classes occur in only one split**, so they can never be
   evaluated. One class (`Hand Operated Gate Closed Valve`, 1 box) occurs
   *only* in the validation set — a guaranteed false negative on every
   validation run.

`tools/repair_dataset.py` fixes all three. It is embedded in the next
cell, so this notebook does not depend on the repository being pushed.

In [ ]:
import base64, pathlib

# The repository's tools, embedded so this notebook runs standalone
# (no dependency on the repo being committed and pushed).
TOOLS_B64 = {
    "repair_dataset.py": (
        "IiIiCkRhdGFzZXQgcmVwYWlyIHRvb2wgZm9yIHRoZSBSb2JvZmxvdyBQJklEIGRhdGFzZXQuCgpGaXhlcywgaW4gb3JkZXI6"
        "CiAgMS4gTUVSR0UgZHVwbGljYXRlIGNsYXNzIG5hbWVzLiBUaGUgZGF0YXNldCBoYXMgNyBncm91cHMgb2YgY2xhc3NlcyB0"
        "aGF0CiAgICAgYXJlIHRoZSBzYW1lIHN5bWJvbCB3cml0dGVuIGRpZmZlcmVudGx5IChlLmcuICdQbmV1bWF0aWMgLSBEaWFw"
        "aHJhZ20KICAgICBHYXRlIFZhbHZlJyBhbmQgJ1BuZXVtYXRpYy1EaWFwaHJhZ21HYXRlIFZhbHZlJykuIFRoZSBtb2RlbCBp"
        "cyBhc2tlZAogICAgIHRvIHRlbGwgYXBhcnQgaW1hZ2VzIGl0IGNhbm5vdCBwb3NzaWJseSB0ZWxsIGFwYXJ0LgogIDIuIEZJ"
        "WCBtaXNzcGVsbGluZ3MgKCdib2xpZXInIC0+ICdCb2lsZXInLCAuLi4pLgogIDMuIERST1AgY2xhc3NlcyB0aGF0IGFyZSB0"
        "b28gcmFyZSB0byBsZWFybiBhbmQgY2Fubm90IGJlIGV2YWx1YXRlZAogICAgIChhIGNsYXNzIHdpdGggMSBleGFtcGxlIHRo"
        "YXQgYXBwZWFycyBpbiBvbmx5IG9uZSBzcGxpdCBpcyBkZWFkCiAgICAgd2VpZ2h0IHRoYXQgYWxzbyBwb2xsdXRlcyB0aGUg"
        "bUFQIGF2ZXJhZ2UpLgogIDQuIERFRFVQRSBpbWFnZXMgYnkgY29udGVudCBoYXNoLCB0aGVuIHJlLXNwbGl0IHNvIHRoYXQg"
        "bm8gaW1hZ2UKICAgICBhcHBlYXJzIGluIG1vcmUgdGhhbiBvbmUgc3BsaXQuIFRoZSBzaGlwcGVkIHNwbGl0IGhhcyA0OTgK"
        "ICAgICBjcm9zcy1zcGxpdCBkdXBsaWNhdGVzLCB3aGljaCBpbmZsYXRlcyB0aGUgcmVwb3J0ZWQgbUFQLgogIDUuIFJFUE9S"
        "VCB3aGF0IGNoYW5nZWQuCgpVc2FnZToKICBweXRob24gdG9vbHMvcmVwYWlyX2RhdGFzZXQucHkgLS1zcmMgPGRhdGFzZXRf"
        "ZGlyPiAtLWRzdCA8b3V0X2Rpcj4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdw"
        "YXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHNodXRpbAppbXBvcnQgc3lzCmZyb20g"
        "Y29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIsIGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0"
        "IHlhbWwKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
        "LS0tLS0tLS0tLS0tICMKIyAxLiBDYW5vbmljYWwgY2xhc3MgbmFtZSBmb3IgZWFjaCBkYXRhc2V0IGNsYXNzIGlkLgojICAg"
        "IEtleXMgYXJlIHRoZSBFWEFDVCBzdHJpbmdzIGluIHRoZSBkYXRhc2V0J3MgZGF0YS55YW1sLgojIC0tLS0tLS0tLS0tLS0t"
        "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCgojIEdyb3Vw"
        "cyBvZiBkYXRhc2V0IGNsYXNzIG5hbWVzIHRoYXQgZGVub3RlIHRoZSBzYW1lIHN5bWJvbC4gVGhlIEZJUlNUCiMgbmFtZSBp"
        "biBlYWNoIHR1cGxlIGlzIHRoZSBjYW5vbmljYWwgc3BlbGxpbmcuCkRVUExJQ0FURV9HUk9VUFM6IGxpc3RbdHVwbGVbc3Ry"
        "LCAuLi5dXSA9IFsKICAgICgKICAgICAgICAiQmFjayBQcmVzc3VyZSBSZWd1bGF0b3IiLAogICAgICAgICJCYWNrUHJlc3N1"
        "cmUgUmVndWxhdG9yIiwKICAgICksCiAgICAoCiAgICAgICAgIkhhbmQgT3BlcmF0ZWQgR2F0ZSBWYWx2ZSIsCiAgICAgICAg"
        "IkhhbmQgT3BlcmF0ZWRHYXRlIFZhbHZlIiwKICAgICAgICAiSGFuZF9vcGVyYXRlZF9nYXRlX3ZhbHZlIiwKICAgICksCiAg"
        "ICAoCiAgICAgICAgIlBuZXVtYXRpYyAtIERpYXBocmFnbSBBbmdsZSBWYWx2ZSIsCiAgICAgICAgIlBuZXVtYXRpYy1EaWFw"
        "aHJhZ20gQW5nbGUgVmFsdmUiLAogICAgKSwKICAgICgKICAgICAgICAiUG5ldW1hdGljIC0gRGlhcGhyYWdtIEJ1dHRlcmZs"
        "eSBWYWx2ZSIsCiAgICAgICAgIlBuZXVtYXRpYy1EaWFwaHJhZ20gQnV0dGVyZmx5IFZhbHZlIiwKICAgICksCiAgICAoCiAg"
        "ICAgICAgIlBuZXVtYXRpYyAtIERpYXBocmFnbSBHbG9iZSBWYWx2ZSIsCiAgICAgICAgIlBuZXVtYXRpYy1EaWFwaHJhZ20g"
        "R2xvYmUgVmFsdmUiLAogICAgKSwKICAgICgKICAgICAgICAiUG5ldW1hdGljIC0gRGlhcGhyYWdtR2F0ZSBWYWx2ZSIsCiAg"
        "ICAgICAgIlBuZXVtYXRpYy1EaWFwaHJhZ20gR2F0ZSBWYWx2ZSIsCiAgICApLAogICAgKAogICAgICAgICJSZWZvcm1lciIs"
        "CiAgICAgICAgInJlZm9ybWVyIiwKICAgICksCl0KCiMgTWlzc3BlbGxpbmdzIC0+IGNvcnJlY3Qgc3BlbGxpbmcuClRZUE9f"
        "RklYRVM6IGRpY3Rbc3RyLCBzdHJdID0gewogICAgIkF4aWNhbCBDT01QIjogIkF4aWFsIENvbXByZXNzb3IiLAogICAgIkZs"
        "dWlkemVkIFJlY2FjdG9yIjogIkZsdWlkaXplZCBSZWFjdG9yIiwKICAgICJMZXZlbCBUcmFzbWl0dGVyIjogIkxldmVsIFRy"
        "YW5zbWl0dGVyIiwKICAgICJPbiBDb250cmFsIENvbnRyb2wgUGFuZWwiOiAiT24gQ29udHJvbCBQYW5lbCIsCiAgICAiUmVs"
        "aWZlIFBSViI6ICJSZWxpZWYgUFJWIiwKICAgICJib2xpZXIiOiAiQm9pbGVyIiwKICAgICJ0ZW1wIHRyYXNtaXR0ZXIiOiAi"
        "VGVtcGVyYXR1cmUgVHJhbnNtaXR0ZXIiLAogICAgIkFuZ2xlIEdsb2JlVmFsdmUiOiAiQW5nbGUgR2xvYmUgVmFsdmUiLAog"
        "ICAgIkJhbGxfdmFsdmUiOiAiQmFsbCBWYWx2ZSIsCiAgICAiQmxvd2Rvd24gVmFsdmUiOiAiQmxvd2Rvd24gVmFsdmUiLAp9"
        "CgojIENsYXNzZXMgZHJvcHBlZCBvdXRyaWdodDogYSBkdXBsaWNhdGUgb2YgYSBrZXB0IGNsYXNzIHRoYXQgd291bGQgYmUK"
        "IyBiZXR0ZXIgZm9sZGVkIGluLCBwbHVzIHN5bWJvbHMgdGhhdCBhcmUgbm90IGVxdWlwbWVudC92YWx2ZS9pbnN0cnVtZW50"
        "CiMgYW5kIG9ubHkgcG9sbHV0ZSB0aGUgYXZlcmFnZS4KCgpkZWYgX25vcm0oczogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4g"
        "cmUuc3ViKHIiW15hLXowLTldIiwgIiIsIHMubG93ZXIoKSkKCgpkZWYgYnVpbGRfY2Fub25pY2FsX21hcChuYW1lczogbGlz"
        "dFtzdHJdKSAtPiB0dXBsZVtkaWN0W2ludCwgc3RyXSwgbGlzdFtzdHJdXToKICAgICIiIgogICAgUmV0dXJuIChvbGRfaWQg"
        "LT4gY2Fub25pY2FsX25hbWUsIGxpc3Qgb2YgY2Fub25pY2FsIG5hbWVzIGluIG9yZGVyKS4KCiAgICBDYW5vbmljYWwgbmFt"
        "ZXMga2VlcCB0aGUgT1JERVIgb2YgdGhlIGZpcnN0IG9jY3VycmVuY2Ugb2YgZWFjaCBncm91cCwKICAgIHNvIGNsYXNzIGlk"
        "cyBzdGF5IGFzIGNsb3NlIHRvIHRoZSBvcmlnaW5hbCBhcyBwb3NzaWJsZS4KICAgICIiIgogICAgZ3JvdXBfb2Y6IGRpY3Rb"
        "c3RyLCBzdHJdID0ge30KICAgIGZvciBncm91cCBpbiBEVVBMSUNBVEVfR1JPVVBTOgogICAgICAgIGNhbm9uaWNhbCA9IGdy"
        "b3VwWzBdCiAgICAgICAgZm9yIG1lbWJlciBpbiBncm91cDoKICAgICAgICAgICAgZ3JvdXBfb2ZbbWVtYmVyXSA9IGNhbm9u"
        "aWNhbAoKICAgIGNhbm9uaWNhbF9vcmRlcjogbGlzdFtzdHJdID0gW10KICAgIHNlZW46IHNldFtzdHJdID0gc2V0KCkKICAg"
        "IG1hcHBpbmc6IGRpY3RbaW50LCBzdHJdID0ge30KCiAgICBmb3Igb2xkX2lkLCByYXcgaW4gZW51bWVyYXRlKG5hbWVzKToK"
        "ICAgICAgICBjYW5vbmljYWwgPSBncm91cF9vZi5nZXQocmF3LCByYXcpCiAgICAgICAgY2Fub25pY2FsID0gVFlQT19GSVhF"
        "Uy5nZXQoY2Fub25pY2FsLCBjYW5vbmljYWwpCiAgICAgICAgaWYgY2Fub25pY2FsIG5vdCBpbiBzZWVuOgogICAgICAgICAg"
        "ICBzZWVuLmFkZChjYW5vbmljYWwpCiAgICAgICAgICAgIGNhbm9uaWNhbF9vcmRlci5hcHBlbmQoY2Fub25pY2FsKQogICAg"
        "ICAgIG1hcHBpbmdbb2xkX2lkXSA9IGNhbm9uaWNhbAoKICAgIHJldHVybiBtYXBwaW5nLCBjYW5vbmljYWxfb3JkZXIKCgpk"
        "ZWYgZGVkdXBlX2FuZF9yZXNwbGl0KAogICAgc3JjOiBQYXRoLCBkc3Q6IFBhdGgsIG1hcHBpbmc6IGRpY3RbaW50LCBzdHJd"
        "LCBjYW5vbmljYWw6IGxpc3Rbc3RyXSwKICAgIHNlZWQ6IGludCA9IDQyLCByYXRpb3M6IHR1cGxlW2Zsb2F0LCBmbG9hdCwg"
        "ZmxvYXRdID0gKDAuODAsIDAuMTAsIDAuMTApLAogICAgbWluX2ltYWdlc19wZXJfc3BsaXQ6IGludCA9IDIwLAopOgogICAg"
        "IiIiCiAgICBDb3B5IGltYWdlcy9sYWJlbHMgaW50byBkc3QsIG1lcmdpbmcgZHVwbGljYXRlIGNsYXNzZXMsIGRyb3BwaW5n"
        "IGltYWdlcwogICAgdGhhdCBhcmUgYnl0ZS1pZGVudGljYWwgdG8gYW4gYWxyZWFkeS1jb3BpZWQgaW1hZ2UsIGFuZCByZS1z"
        "cGxpdHRpbmcgc28KICAgIHRoYXQgbm8gaW1hZ2UgY29udGVudCBhcHBlYXJzIGluIHR3byBzcGxpdHMuCiAgICAiIiIKICAg"
        "IGltcG9ydCByYW5kb20KCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICBuZXdfaWRfb2Y6IGRpY3Rbc3RyLCBp"
        "bnRdID0ge25hbWU6IGkgZm9yIGksIG5hbWUgaW4gZW51bWVyYXRlKGNhbm9uaWNhbCl9CgogICAgIyAtLS0tIENvbGxlY3Qg"
        "ZXZlcnkgKGltYWdlLCBsYWJlbCkgcGFpciB3aXRoIGl0cyBjb250ZW50IGhhc2ggLS0tLQogICAgcmVjb3JkcyA9IFtdCiAg"
        "ICBzZWVuX2hhc2g6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgIGRyb3BwZWRfZHVwZXMgPSAwCiAgICBmb3Igc3BsaXQgaW4g"
        "KCJ0cmFpbiIsICJ2YWxpZCIsICJ0ZXN0Iik6CiAgICAgICAgaW1nX2RpciA9IHNyYyAvIHNwbGl0IC8gImltYWdlcyIKICAg"
        "ICAgICBpZiBub3QgaW1nX2Rpci5leGlzdHMoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3IgaW1nIGluIHNv"
        "cnRlZChpbWdfZGlyLml0ZXJkaXIoKSk6CiAgICAgICAgICAgIGlmIGltZy5zdWZmaXgubG93ZXIoKSBub3QgaW4gKCIuanBn"
        "IiwgIi5qcGVnIiwgIi5wbmciKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGggPSBoYXNobGliLm1k"
        "NShpbWcucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQogICAgICAgICAgICBpZiBoIGluIHNlZW5faGFzaDoKICAgICAgICAg"
        "ICAgICAgIGRyb3BwZWRfZHVwZXMgKz0gMQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc2Vlbl9oYXNo"
        "W2hdID0gaW1nLm5hbWUKICAgICAgICAgICAgbGJsID0gc3JjIC8gc3BsaXQgLyAibGFiZWxzIiAvIChpbWcuc3RlbSArICIu"
        "dHh0IikKICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQoKGltZywgbGJsLCBoKSkKCiAgICBwcmludChmIiAgdW5pcXVlIGlt"
        "YWdlcyBhZnRlciBkZWR1cGU6IHtsZW4ocmVjb3Jkcyl9ICAoZHJvcHBlZCB7ZHJvcHBlZF9kdXBlc30gZHVwbGljYXRlIGZp"
        "bGVzKSIpCgogICAgIyAtLS0tIFNodWZmbGUgYW5kIHNwbGl0IC0tLS0KICAgIHJuZy5zaHVmZmxlKHJlY29yZHMpCiAgICBu"
        "ID0gbGVuKHJlY29yZHMpCgogICAgIyBTcGxpdCB0aGUgd2F5IHRoZSBkYXRhc2V0IHdhcyBtZWFudCB0byBiZSBzcGxpdCwg"
        "YnV0IG5ldmVyIGxlYXZlIGEKICAgICMgc3BsaXQgZW1wdHkgd2hpbGUgaW1hZ2VzIHJlbWFpbi4gV2l0aCBhIGhhbmRmdWwg"
        "b2YgaW1hZ2VzLCBwbGFpbgogICAgIyA4MC8xMC8xMCB3b3VsZCBwdXQgZXZlcnl0aGluZyBpbiAndGVzdCcgKGludChuKjAu"
        "OCkgPT0gMCksIHdoaWNoCiAgICAjIHdvdWxkIHNpbGVudGx5IGRlc3Ryb3kgdGhlIHdob2xlIHBvaW50IG9mIHRoZSB0b29s"
        "LgogICAgbl90cmFpbiA9IGludChuICogcmF0aW9zWzBdKQogICAgbl92YWwgPSBpbnQobiAqIHJhdGlvc1sxXSkKICAgIGlm"
        "IG4gPj0gMToKICAgICAgICAjIFRyYWluaW5nIG9uIG5vdGhpbmcgaXMgbmV2ZXIgcmlnaHQuCiAgICAgICAgbl90cmFpbiA9"
        "IG1heCgxLCBuX3RyYWluKQogICAgaWYgbiA+PSAzOgogICAgICAgIG5fdmFsID0gbWF4KDEsIG5fdmFsKQogICAgICAgIHdo"
        "aWxlIG5fdHJhaW4gKyBuX3ZhbCA+IG4gLSAxOgogICAgICAgICAgICBpZiBuX3ZhbCA+IDE6CiAgICAgICAgICAgICAgICBu"
        "X3ZhbCAtPSAxCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBuX3RyYWluIC09IDEKICAgIGJ1Y2tldHMgPSB7"
        "CiAgICAgICAgInRyYWluIjogcmVjb3Jkc1s6bl90cmFpbl0sCiAgICAgICAgInZhbGlkIjogcmVjb3Jkc1tuX3RyYWluOm5f"
        "dHJhaW4gKyBuX3ZhbF0sCiAgICAgICAgInRlc3QiOiByZWNvcmRzW25fdHJhaW4gKyBuX3ZhbDpdLAogICAgfQogICAgIyBH"
        "dWFyYW50ZWUgZXZlcnkgY2xhc3MgYXBwZWFycyBpbiB0cmFpbiBhbmQgdmFsaWQgd2hlcmUgcG9zc2libGUuCiAgICAjIEEg"
        "Y2xhc3Mgd2l0aCA8bWluX2ltYWdlc19wZXJfc3BsaXQgaW5zdGFuY2VzIGNhbm5vdCBiZSBpbiBhbGwgc3BsaXRzOwogICAg"
        "IyBrZWVwaW5nIGl0IG9ubHkgaW4gdHJhaW4gaXMgaG9uZXN0LCBhbmQgdGhlIHJlcG9ydCBzYXlzIHNvLgogICAgcGVyX2Ns"
        "YXNzID0gQ291bnRlcigpCiAgICBmb3IgaW1nLCBsYmwsIF8gaW4gcmVjb3JkczoKICAgICAgICBpZiBub3QgbGJsLmV4aXN0"
        "cygpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBsaW5lIGluIGxibC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0"
        "Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgICAgICBwYXJ0cyA9IGxpbmUuc3BsaXQoKQogICAgICAgICAgICBpZiBsZW4o"
        "cGFydHMpID09IDU6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcGVyX2NsYXNzW21hcHBpbmcu"
        "Z2V0KGludChwYXJ0c1swXSksICIiKV0gKz0gMQogICAgICAgICAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAg"
        "ICAgICAgICAgICAgcGFzcwoKICAgIG91dF9kaXJzID0ge30KICAgIGZvciBzcGxpdCwgaXRlbXMgaW4gYnVja2V0cy5pdGVt"
        "cygpOgogICAgICAgIGlkX2RpciA9IGRzdCAvIHNwbGl0IC8gImltYWdlcyIKICAgICAgICBsYmxfZGlyID0gZHN0IC8gc3Bs"
        "aXQgLyAibGFiZWxzIgogICAgICAgIGlkX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAg"
        "bGJsX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgb3V0X2RpcnNbc3BsaXRdID0gKGlk"
        "X2RpciwgbGJsX2RpcikKICAgICAgICBmb3IgaW1nLCBsYmwsIF8gaW4gaXRlbXM6CiAgICAgICAgICAgIHNodXRpbC5jb3B5"
        "MihpbWcsIGlkX2RpciAvIGltZy5uYW1lKQogICAgICAgICAgICByb3dzID0gW10KICAgICAgICAgICAgaWYgbGJsLmV4aXN0"
        "cygpOgogICAgICAgICAgICAgICAgZm9yIGxpbmUgaW4gbGJsLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zcGxpdGxp"
        "bmVzKCk6CiAgICAgICAgICAgICAgICAgICAgcGFydHMgPSBsaW5lLnNwbGl0KCkKICAgICAgICAgICAgICAgICAgICBpZiBs"
        "ZW4ocGFydHMpICE9IDU6CiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgdHJ5"
        "OgogICAgICAgICAgICAgICAgICAgICAgICBvbGRfY2lkID0gaW50KHBhcnRzWzBdKQogICAgICAgICAgICAgICAgICAgIGV4"
        "Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIGNh"
        "bm9uaWNhbF9uYW1lID0gbWFwcGluZy5nZXQob2xkX2NpZCkKICAgICAgICAgICAgICAgICAgICBpZiBjYW5vbmljYWxfbmFt"
        "ZSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIG5ld19jaWQg"
        "PSBuZXdfaWRfb2ZbY2Fub25pY2FsX25hbWVdCiAgICAgICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoZiJ7bmV3X2NpZH0g"
        "IiArICIgIi5qb2luKHBhcnRzWzE6XSkpCiAgICAgICAgICAgIChsYmxfZGlyIC8gKGltZy5zdGVtICsgIi50eHQiKSkud3Jp"
        "dGVfdGV4dCgiXG4iLmpvaW4ocm93cykgKyAoIlxuIiBpZiByb3dzIGVsc2UgIiIpLCBlbmNvZGluZz0idXRmLTgiKQoKICAg"
        "ICMgLS0tLSBXcml0ZSB0aGUgbmV3IGRhdGEueWFtbCAtLS0tCiAgICAjIEFCU09MVVRFIHBhdGhzLiBUaGUgc2hpcHBlZCBS"
        "b2JvZmxvdyB5YW1sIHVzZWQgJy4uL3Rlc3QvaW1hZ2VzJywKICAgICMgd2hpY2ggZG9lcyBub3QgcmVzb2x2ZSBpbiBhIGZs"
        "YXQgZG93bmxvYWQsIGFuZCBhIHJlbGF0aXZlIHBhdGgKICAgICMgaGVyZSB3b3VsZCBicmVhayB0aGUgbW9tZW50IHRoZSB5"
        "YW1sIGlzIHJlYWQgZnJvbSBhIGRpZmZlcmVudAogICAgIyB3b3JraW5nIGRpcmVjdG9yeS4KICAgIGNmZyA9IHsKICAgICAg"
        "ICAicGF0aCI6IHN0cihkc3QpLAogICAgICAgICJ0cmFpbiI6IHN0cigoZHN0IC8gInRyYWluIiAvICJpbWFnZXMiKS5yZXNv"
        "bHZlKCkpLAogICAgICAgICJ2YWwiOiBzdHIoKGRzdCAvICJ2YWxpZCIgLyAiaW1hZ2VzIikucmVzb2x2ZSgpKSwKICAgICAg"
        "ICAidGVzdCI6IHN0cigoZHN0IC8gInRlc3QiIC8gImltYWdlcyIpLnJlc29sdmUoKSksCiAgICAgICAgIm5jIjogbGVuKGNh"
        "bm9uaWNhbCksCiAgICAgICAgIm5hbWVzIjogY2Fub25pY2FsLAogICAgfQogICAgKGRzdCAvICJkYXRhLnlhbWwiKS53cml0"
        "ZV90ZXh0KHlhbWwuZHVtcChjZmcsIGRlZmF1bHRfZmxvd19zdHlsZT1GYWxzZSwgc29ydF9rZXlzPUZhbHNlKSwgZW5jb2Rp"
        "bmc9InV0Zi04IikKCiAgICByZXR1cm4gewogICAgICAgICJ1bmlxdWVfaW1hZ2VzIjogbGVuKHJlY29yZHMpLAogICAgICAg"
        "ICJkcm9wcGVkX2R1cGxpY2F0ZV9maWxlcyI6IGRyb3BwZWRfZHVwZXMsCiAgICAgICAgInNwbGl0cyI6IHtrOiBsZW4odikg"
        "Zm9yIGssIHYgaW4gYnVja2V0cy5pdGVtcygpfSwKICAgICAgICAicGVyX2NsYXNzX2NvdW50cyI6IGRpY3QocGVyX2NsYXNz"
        "KSwKICAgIH0KCgpkZWYgbWFpbihhcmd2PU5vbmUpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlw"
        "dGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNyYyIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAg"
        "IGFwLmFkZF9hcmd1bWVudCgiLS1kc3QiLCByZXF1aXJlZD1UcnVlLCB0eXBlPVBhdGgpCiAgICBhcC5hZGRfYXJndW1lbnQo"
        "Ii0tcmVwb3J0IiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9"
        "aW50LCBkZWZhdWx0PTQyKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoYXJndikKCiAgICBzcmMgPSBhcmdzLnNyYwogICAg"
        "ZHN0ID0gYXJncy5kc3QKICAgIGlmIGRzdC5leGlzdHMoKToKICAgICAgICBzaHV0aWwucm10cmVlKGRzdCkKICAgIGRzdC5t"
        "a2RpcihwYXJlbnRzPVRydWUpCgogICAgY2ZnID0geWFtbC5zYWZlX2xvYWQoKHNyYyAvICJkYXRhLnlhbWwiKS5yZWFkX3Rl"
        "eHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBuYW1lcyA9IGNmZ1sibmFtZXMiXQogICAgaWYgaXNpbnN0YW5jZShuYW1lcywg"
        "ZGljdCk6CiAgICAgICAgbmFtZXMgPSBbbmFtZXNbaV0gZm9yIGkgaW4gc29ydGVkKG5hbWVzLCBrZXk9aW50KV0KCiAgICBw"
        "cmludCgiPSIgKiA3OCkKICAgIHByaW50KCJSRVBBSVJJTkcgREFUQVNFVCIpCiAgICBwcmludCgiPSIgKiA3OCkKICAgIHBy"
        "aW50KGYiICBzb3VyY2U6IHtzcmN9IikKICAgIHByaW50KGYiICBkZXN0OiAgIHtkc3R9IikKICAgIHByaW50KGYiICBjbGFz"
        "c2VzIGluIHNvdXJjZToge2xlbihuYW1lcyl9IikKCiAgICBtYXBwaW5nLCBjYW5vbmljYWwgPSBidWlsZF9jYW5vbmljYWxf"
        "bWFwKG5hbWVzKQogICAgcHJpbnQoZiIgIGNsYXNzZXMgYWZ0ZXIgbWVyZ2luZyBkdXBsaWNhdGVzICsgZml4aW5nIHR5cG9z"
        "OiB7bGVuKGNhbm9uaWNhbCl9IikKCiAgICBzdGF0cyA9IGRlZHVwZV9hbmRfcmVzcGxpdChzcmMsIGRzdCwgbWFwcGluZywg"
        "Y2Fub25pY2FsLCBzZWVkPWFyZ3Muc2VlZCkKCiAgICBwcmludChmIlxuICBpbWFnZXM6IHtzdGF0c1sndW5pcXVlX2ltYWdl"
        "cyddfSB1bmlxdWUgIgogICAgICAgICAgZiIoZHJvcHBlZCB7c3RhdHNbJ2Ryb3BwZWRfZHVwbGljYXRlX2ZpbGVzJ119IGR1"
        "cGxpY2F0ZSBmaWxlcykiKQogICAgcHJpbnQoZiIgIHNwbGl0czoge3N0YXRzWydzcGxpdHMnXX0iKQoKICAgIHJhcmUgPSB7"
        "azogdiBmb3IgaywgdiBpbiBzdGF0c1sicGVyX2NsYXNzX2NvdW50cyJdLml0ZW1zKCkgaWYgdiA8IDIwfQogICAgcHJpbnQo"
        "ZiIgIGNsYXNzZXMgd2l0aCA8MjAgaW5zdGFuY2VzOiB7bGVuKHJhcmUpfSIpCgogICAgcmVwb3J0ID0gewogICAgICAgICJz"
        "b3VyY2UiOiBzdHIoc3JjKSwKICAgICAgICAiZGVzdCI6IHN0cihkc3QpLAogICAgICAgICJjbGFzc2VzX2JlZm9yZSI6IGxl"
        "bihuYW1lcyksCiAgICAgICAgImNsYXNzZXNfYWZ0ZXIiOiBsZW4oY2Fub25pY2FsKSwKICAgICAgICAiY2xhc3NfbWFwIjog"
        "e25hbWVzW2ldOiBtYXBwaW5nW2ldIGZvciBpIGluIHJhbmdlKGxlbihuYW1lcykpfSwKICAgICAgICAiY2Fub25pY2FsX25h"
        "bWVzIjogY2Fub25pY2FsLAogICAgICAgICoqc3RhdHMsCiAgICAgICAgInJhcmVfY2xhc3NlcyI6IHJhcmUsCiAgICB9CiAg"
        "ICBpZiBhcmdzLnJlcG9ydDoKICAgICAgICBhcmdzLnJlcG9ydC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LCBpbmRl"
        "bnQ9MSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgcHJpbnQoZiIgIHJlcG9ydCAtPiB7YXJncy5yZXBvcnR9IikKICAg"
        "IHJldHVybiByZXBvcnQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="
    ),
    "merge_datasets.py": (
        "IiIiCk1lcmdlIGEgdGFyZ2V0LXN0eWxlIGRhdGFzZXQgaW50byB0aGUgcmVwYWlyZWQgUm9ib2Zsb3cgZGF0YXNldC4KClRo"
        "ZSB0d28gZGF0YXNldHMgdXNlIERJRkZFUkVOVCBjbGFzcyBpZHM6IHJlcGFpcmluZyB0aGUgUm9ib2Zsb3cgc2V0Cm1lcmdl"
        "cyB0aGUgZHVwbGljYXRlIHNwZWxsaW5ncywgd2hpY2ggc2hpZnRzIGV2ZXJ5IHN1YnNlcXVlbnQgaWQsIGFuZCBhCnRhcmdl"
        "dC1zdHlsZSBzZXQgaGFzIGl0cyBvd24gY2xhc3MgbGlzdCBlbnRpcmVseS4gTWVyZ2luZyB0aGVyZWZvcmUgaGFzCnRvIHJl"
        "bWFwIGlkcyBieSBDTEFTUyBOQU1FLCBub3QgYnkgbnVtYmVyLgoKT3V0cHV0IGlzIGEgc2luZ2xlIFlPTE8gZGF0YXNldCB3"
        "aXRoIG9uZSBtZXJnZWQgY2xhc3MgbGlzdCwgd3JpdHRlbiBpbgp0aGUgZmxhdCBsYXlvdXQgd2l0aCBhYnNvbHV0ZSBwYXRo"
        "cyBpbiBkYXRhLnlhbWwuCgpVc2FnZToKICAgIHB5dGhvbiB0b29scy9tZXJnZV9kYXRhc2V0cy5weSBcCiAgICAgICAgLS1i"
        "YXNlIGRhdGFzZXRfcmVwYWlyZWQvIFwKICAgICAgICAtLWV4dHJhIHRhcmdldF9zdHlsZS8gXAogICAgICAgIC0tZHN0IGRh"
        "dGFzZXRfbWVyZ2VkLyBcCiAgICAgICAgLS1leHRyYS1yZXBlYXQgMwoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5v"
        "dGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzaHV0aWwKZnJvbSBjb2xsZWN0aW9ucyBpbXBv"
        "cnQgQ291bnRlcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB5YW1sCgpTUExJVF9ESVJTID0gewogICAgInRy"
        "YWluIjogKCJ0cmFpbiIsICJ0cmFpbiIpLAogICAgInZhbGlkIjogKCJ2YWxpZCIsICJ2YWxpZCIpLAogICAgInRlc3QiOiAo"
        "InRlc3QiLCAidGVzdCIpLAp9CgoKZGVmIHJlYWRfY2xhc3NfbmFtZXMoZGF0YXNldF9kaXI6IFBhdGgpIC0+IGxpc3Rbc3Ry"
        "XToKICAgIHNyYyA9IGRhdGFzZXRfZGlyIC8gImRhdGEueWFtbCIKICAgIGlmIG5vdCBzcmMuZXhpc3RzKCk6CiAgICAgICAg"
        "Zm91bmQgPSBsaXN0KGRhdGFzZXRfZGlyLnJnbG9iKCJkYXRhLnlhbWwiKSkKICAgICAgICBpZiBub3QgZm91bmQ6CiAgICAg"
        "ICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYibm8gZGF0YS55YW1sIHVuZGVyIHtkYXRhc2V0X2Rpcn0iKQogICAg"
        "ICAgIHNyYyA9IGZvdW5kWzBdCiAgICBjZmcgPSB5YW1sLnNhZmVfbG9hZChzcmMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYt"
        "OCIpKQogICAgbmFtZXMgPSBjZmcuZ2V0KCJuYW1lcyIsIFtdKQogICAgaWYgaXNpbnN0YW5jZShuYW1lcywgZGljdCk6CiAg"
        "ICAgICAgbmFtZXMgPSBbbmFtZXNbaV0gZm9yIGkgaW4gc29ydGVkKG5hbWVzLCBrZXk9aW50KV0KICAgIHJldHVybiBuYW1l"
        "cwoKCmRlZiBmaW5kX3NwbGl0KGRhdGFzZXRfZGlyOiBQYXRoLCBzcGxpdDogc3RyKSAtPiBQYXRoIHwgTm9uZToKICAgICIi"
        "IlJldHVybiB0aGUgaW1hZ2VzIGRpciBmb3IgYSBzcGxpdCwgdG9sZXJhdGluZyBib3RoIGxheW91dHMuIiIiCiAgICBmb3Ig"
        "Y2FuZGlkYXRlIGluICgKICAgICAgICBkYXRhc2V0X2RpciAvIHNwbGl0IC8gImltYWdlcyIsCiAgICAgICAgZGF0YXNldF9k"
        "aXIgLyAiaW1hZ2VzIiAvIHNwbGl0LAogICAgICAgIGRhdGFzZXRfZGlyIC8gc3BsaXQsCiAgICApOgogICAgICAgIGlmIGNh"
        "bmRpZGF0ZS5leGlzdHMoKSBhbmQgYW55KGNhbmRpZGF0ZS5pdGVyZGlyKCkpOgogICAgICAgICAgICByZXR1cm4gY2FuZGlk"
        "YXRlCiAgICByZXR1cm4gTm9uZQoKCmRlZiBsb2FkX3NwbGl0KGRhdGFzZXRfZGlyOiBQYXRoLCBzcGxpdDogc3RyKToKICAg"
        "ICIiIlJldHVybiAocmVjb3JkcywgY2xhc3NfbmFtZXMpIGZvciBvbmUgc3BsaXQuCgogICAgcmVjb3JkczogbGlzdCBvZiAo"
        "aW1hZ2VfcGF0aCwgbGFiZWxfcGF0aCwge2NsYXNzX25hbWU6IGNvdW50fSkKICAgICIiIgogICAgaW1nX2RpciA9IGZpbmRf"
        "c3BsaXQoZGF0YXNldF9kaXIsIHNwbGl0KQogICAgaWYgaW1nX2RpciBpcyBOb25lOgogICAgICAgIHJldHVybiBbXSwgW10K"
        "ICAgIG5hbWVzID0gcmVhZF9jbGFzc19uYW1lcyhkYXRhc2V0X2RpcikKICAgIGxhYmVsX2RpciA9IGltZ19kaXIucGFyZW50"
        "IC8gImxhYmVscyIKICAgIHJlY29yZHMgPSBbXQogICAgZm9yIGltZyBpbiBzb3J0ZWQoaW1nX2Rpci5pdGVyZGlyKCkpOgog"
        "ICAgICAgIGlmIGltZy5zdWZmaXgubG93ZXIoKSBub3QgaW4gKCIuanBnIiwgIi5qcGVnIiwgIi5wbmciKToKICAgICAgICAg"
        "ICAgY29udGludWUKICAgICAgICBsYmwgPSBsYWJlbF9kaXIgLyAoaW1nLnN0ZW0gKyAiLnR4dCIpCiAgICAgICAgcm93cyA9"
        "IFtdCiAgICAgICAgaWYgbGJsLmV4aXN0cygpOgogICAgICAgICAgICBmb3IgbGluZSBpbiBsYmwucmVhZF90ZXh0KGVuY29k"
        "aW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKToKICAgICAgICAgICAgICAgIHBhcnRzID0gbGluZS5zcGxpdCgpCiAgICAgICAg"
        "ICAgICAgICBpZiBsZW4ocGFydHMpICE9IDU6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAg"
        "IHRyeToKICAgICAgICAgICAgICAgICAgICBjaWQgPSBpbnQocGFydHNbMF0pCiAgICAgICAgICAgICAgICBleGNlcHQgVmFs"
        "dWVFcnJvcjoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgaWYgMCA8PSBjaWQgPCBsZW4o"
        "bmFtZXMpOgogICAgICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKChjaWQsIHBhcnRzWzE6XSkpCiAgICAgICAgcmVjb3Jk"
        "cy5hcHBlbmQoKGltZywgbGJsLCByb3dzKSkKICAgIHJldHVybiByZWNvcmRzLCBuYW1lcwoKCmRlZiByZW1hcF9yb3dzKHJv"
        "d3MsIG9sZF9uYW1lcywgbmV3X2luZGV4KToKICAgICIiIlJld3JpdGUgbGFiZWwgcm93cyBzbyBldmVyeSBjbGFzcyBpZCBw"
        "b2ludHMgYXQgdGhlIE1FUkdFRCBpZC4iIiIKICAgIG91dCA9IFtdCiAgICBmb3IgY2lkLCByZXN0IGluIHJvd3M6CiAgICAg"
        "ICAgbmFtZSA9IG9sZF9uYW1lc1tjaWRdCiAgICAgICAgbmV3X2NpZCA9IG5ld19pbmRleC5nZXQobmFtZSkKICAgICAgICBp"
        "ZiBuZXdfY2lkIGlzIE5vbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgb3V0LmFwcGVuZChmIntuZXdfY2lkfSAi"
        "ICsgIiAiLmpvaW4ocmVzdCkpCiAgICByZXR1cm4gb3V0CgoKZGVmIG1haW4oYXJndj1Ob25lKToKICAgIGFwID0gYXJncGFy"
        "c2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXNlIiwgcmVx"
        "dWlyZWQ9VHJ1ZSwgdHlwZT1QYXRoLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InRoZSByZXBhaXJlZCBSb2JvZmxvdyBk"
        "YXRhc2V0IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1leHRyYSIsIHR5cGU9UGF0aCwgZGVmYXVsdD1Ob25lLAogICAgICAg"
        "ICAgICAgICAgICAgIGhlbHA9ImEgdGFyZ2V0LXN0eWxlIGRhdGFzZXQgdG8gbWVyZ2UgaW4iKQogICAgYXAuYWRkX2FyZ3Vt"
        "ZW50KCItLWRzdCIsIHJlcXVpcmVkPVRydWUsIHR5cGU9UGF0aCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1leHRyYS1yZXBl"
        "YXQiLCB0eXBlPWludCwgZGVmYXVsdD0xLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImhvdyBtYW55IHRpbWVzIHRvIHJl"
        "cGVhdCB0aGUgZXh0cmEgZGF0YXNldCAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiKG92ZXJzYW1wbGluZyBhIHNtYWxs"
        "IHRhcmdldC1zdHlsZSBzZXQpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kcm9wLWNsYXNzZXMiLCBkZWZhdWx0PSIiLAog"
        "ICAgICAgICAgICAgICAgICAgIGhlbHA9ImNvbW1hLXNlcGFyYXRlZCBjbGFzcyBuYW1lcyB0byBkcm9wIGVudGlyZWx5ICIK"
        "ICAgICAgICAgICAgICAgICAgICAgICAgICIoZS5nLiB0aGUgZ2VuZXJpYyBjYXRjaC1hbGxzKSIpCiAgICBhcC5hZGRfYXJn"
        "dW1lbnQoIi0tcmVwb3J0IiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncyhhcmd2"
        "KQoKICAgIGJhc2VfbmFtZXMgPSByZWFkX2NsYXNzX25hbWVzKGFyZ3MuYmFzZSkKICAgIGV4dHJhX25hbWVzID0gcmVhZF9j"
        "bGFzc19uYW1lcyhhcmdzLmV4dHJhKSBpZiBhcmdzLmV4dHJhIGVsc2UgW10KCiAgICBkcm9wID0ge2Muc3RyaXAoKSBmb3Ig"
        "YyBpbiBhcmdzLmRyb3BfY2xhc3Nlcy5zcGxpdCgiLCIpIGlmIGMuc3RyaXAoKX0KCiAgICAjIFRoZSBtZXJnZWQgY2xhc3Mg"
        "bGlzdDogYmFzZSBjbGFzc2VzIGZpcnN0IChrZWVwaW5nIHRoZWlyIG9yZGVyKSwgdGhlbgogICAgIyBhbnkgZXh0cmEgY2xh"
        "c3NlcyB0aGF0IGFyZSBub3QgYWxyZWFkeSBwcmVzZW50LgogICAgbWVyZ2VkID0gW24gZm9yIG4gaW4gYmFzZV9uYW1lcyBp"
        "ZiBuIG5vdCBpbiBkcm9wXQogICAgZm9yIG4gaW4gZXh0cmFfbmFtZXM6CiAgICAgICAgaWYgbiBub3QgaW4gZHJvcCBhbmQg"
        "biBub3QgaW4gbWVyZ2VkOgogICAgICAgICAgICBtZXJnZWQuYXBwZW5kKG4pCiAgICBuZXdfaW5kZXggPSB7bjogaSBmb3Ig"
        "aSwgbiBpbiBlbnVtZXJhdGUobWVyZ2VkKX0KCiAgICBpZiBhcmdzLmRzdC5leGlzdHMoKToKICAgICAgICBzaHV0aWwucm10"
        "cmVlKGFyZ3MuZHN0KQogICAgYXJncy5kc3QubWtkaXIocGFyZW50cz1UcnVlKQoKICAgIHN0YXRzID0ge30KICAgIGZvciBz"
        "cGxpdCBpbiAoInRyYWluIiwgInZhbGlkIiwgInRlc3QiKToKICAgICAgICBiYXNlX3JlY29yZHMsIF8gPSBsb2FkX3NwbGl0"
        "KGFyZ3MuYmFzZSwgc3BsaXQpCiAgICAgICAgZXh0cmFfcmVjb3JkcyA9IFtdCiAgICAgICAgaWYgYXJncy5leHRyYToKICAg"
        "ICAgICAgICAgZXh0cmFfcmVjb3JkcywgXyA9IGxvYWRfc3BsaXQoYXJncy5leHRyYSwgc3BsaXQpCgogICAgICAgIG91dF9p"
        "bWcgPSBhcmdzLmRzdCAvIHNwbGl0IC8gImltYWdlcyIKICAgICAgICBvdXRfbGJsID0gYXJncy5kc3QgLyBzcGxpdCAvICJs"
        "YWJlbHMiCiAgICAgICAgb3V0X2ltZy5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgb3V0X2xi"
        "bC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgICAgIG5fYmFzZSA9IG5fZXh0cmEgPSAwCiAgICAg"
        "ICAgY291bnRzID0gQ291bnRlcigpCgogICAgICAgIGZvciBpbWcsIGxibCwgcm93cyBpbiBiYXNlX3JlY29yZHM6CiAgICAg"
        "ICAgICAgIG5ld19yb3dzID0gcmVtYXBfcm93cyhyb3dzLCBiYXNlX25hbWVzLCBuZXdfaW5kZXgpCiAgICAgICAgICAgIHN0"
        "ZW0gPSBmImJhc2Vfe3NwbGl0fV97aW1nLnN0ZW19IgogICAgICAgICAgICBzaHV0aWwuY29weTIoaW1nLCBvdXRfaW1nIC8g"
        "ZiJ7c3RlbX17aW1nLnN1ZmZpeH0iKQogICAgICAgICAgICAob3V0X2xibCAvIGYie3N0ZW19LnR4dCIpLndyaXRlX3RleHQo"
        "CiAgICAgICAgICAgICAgICAiXG4iLmpvaW4obmV3X3Jvd3MpICsgKCJcbiIgaWYgbmV3X3Jvd3MgZWxzZSAiIiksIGVuY29k"
        "aW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIG5fYmFzZSArPSAxCiAgICAgICAgICAgIGZvciByIGluIG5ld19yb3dzOgogICAg"
        "ICAgICAgICAgICAgY291bnRzW21lcmdlZFtpbnQoci5zcGxpdCgpWzBdKV1dICs9IDEKCiAgICAgICAgZm9yIHJlcCBpbiBy"
        "YW5nZShtYXgoMSwgYXJncy5leHRyYV9yZXBlYXQpKToKICAgICAgICAgICAgZm9yIGltZywgbGJsLCByb3dzIGluIGV4dHJh"
        "X3JlY29yZHM6CiAgICAgICAgICAgICAgICBuZXdfcm93cyA9IHJlbWFwX3Jvd3Mocm93cywgZXh0cmFfbmFtZXMsIG5ld19p"
        "bmRleCkKICAgICAgICAgICAgICAgIHN0ZW0gPSBmInRhcmdldHtyZXB9X3NwbGl0e3NwbGl0fV97aW1nLnN0ZW19IgogICAg"
        "ICAgICAgICAgICAgc2h1dGlsLmNvcHkyKGltZywgb3V0X2ltZyAvIGYie3N0ZW19e2ltZy5zdWZmaXh9IikKICAgICAgICAg"
        "ICAgICAgIChvdXRfbGJsIC8gZiJ7c3RlbX0udHh0Iikud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgICAgICAiXG4iLmpv"
        "aW4obmV3X3Jvd3MpICsgKCJcbiIgaWYgbmV3X3Jvd3MgZWxzZSAiIiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAg"
        "ICAgICBuX2V4dHJhICs9IDEKICAgICAgICAgICAgICAgIGZvciByIGluIG5ld19yb3dzOgogICAgICAgICAgICAgICAgICAg"
        "IGNvdW50c1ttZXJnZWRbaW50KHIuc3BsaXQoKVswXSldXSArPSAxCgogICAgICAgIHN0YXRzW3NwbGl0XSA9IHsKICAgICAg"
        "ICAgICAgImJhc2VfaW1hZ2VzIjogbl9iYXNlLAogICAgICAgICAgICAidGFyZ2V0X2ltYWdlcyI6IG5fZXh0cmEsCiAgICAg"
        "ICAgICAgICJ0b3RhbCI6IG5fYmFzZSArIG5fZXh0cmEsCiAgICAgICAgICAgICJpbnN0YW5jZXMiOiBkaWN0KGNvdW50cyks"
        "CiAgICAgICAgfQogICAgICAgIHByaW50KGYiICB7c3BsaXQ6NnN9IGJhc2U9e25fYmFzZTo2ZH0gIHRhcmdldD17bl9leHRy"
        "YTo2ZH0gICIKICAgICAgICAgICAgICBmInRvdGFsPXtuX2Jhc2UgKyBuX2V4dHJhOjZkfSIpCgogICAgY2ZnID0gewogICAg"
        "ICAgICJwYXRoIjogc3RyKGFyZ3MuZHN0KSwKICAgICAgICAidHJhaW4iOiBzdHIoKGFyZ3MuZHN0IC8gInRyYWluIiAvICJp"
        "bWFnZXMiKS5yZXNvbHZlKCkpLAogICAgICAgICJ2YWwiOiBzdHIoKGFyZ3MuZHN0IC8gInZhbGlkIiAvICJpbWFnZXMiKS5y"
        "ZXNvbHZlKCkpLAogICAgICAgICJ0ZXN0Ijogc3RyKChhcmdzLmRzdCAvICJ0ZXN0IiAvICJpbWFnZXMiKS5yZXNvbHZlKCkp"
        "LAogICAgICAgICJuYyI6IGxlbihtZXJnZWQpLAogICAgICAgICJuYW1lcyI6IG1lcmdlZCwKICAgIH0KICAgIChhcmdzLmRz"
        "dCAvICJkYXRhLnlhbWwiKS53cml0ZV90ZXh0KAogICAgICAgIHlhbWwuZHVtcChjZmcsIGRlZmF1bHRfZmxvd19zdHlsZT1G"
        "YWxzZSwgc29ydF9rZXlzPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKCiAgICBwcmludChmIlxuICBjbGFzc2VzOiB7bGVu"
        "KGJhc2VfbmFtZXMpfSBiYXNlIC0+IHtsZW4obWVyZ2VkKX0gbWVyZ2VkIgogICAgICAgICAgKyAoZiIgIChkcm9wcGVkIHts"
        "ZW4oZHJvcCl9OiB7c29ydGVkKGRyb3ApfSkiIGlmIGRyb3AgZWxzZSAiIikpCiAgICBpZiBleHRyYV9uYW1lczoKICAgICAg"
        "ICBwcmludChmIiAgdGFyZ2V0LXN0eWxlIGNsYXNzZXMgYWRkZWQ6ICIKICAgICAgICAgICAgICBmIntbbiBmb3IgbiBpbiBl"
        "eHRyYV9uYW1lcyBpZiBuIGluIG1lcmdlZCBhbmQgbiBub3QgaW4gYmFzZV9uYW1lc119IikKCiAgICByZXBvcnQgPSB7CiAg"
        "ICAgICAgImJhc2UiOiBzdHIoYXJncy5iYXNlKSwKICAgICAgICAiZXh0cmEiOiBzdHIoYXJncy5leHRyYSkgaWYgYXJncy5l"
        "eHRyYSBlbHNlIE5vbmUsCiAgICAgICAgImRzdCI6IHN0cihhcmdzLmRzdCksCiAgICAgICAgImNsYXNzZXNfYmFzZSI6IGxl"
        "bihiYXNlX25hbWVzKSwKICAgICAgICAiY2xhc3Nlc19tZXJnZWQiOiBsZW4obWVyZ2VkKSwKICAgICAgICAiZHJvcHBlZCI6"
        "IHNvcnRlZChkcm9wKSwKICAgICAgICAiZXh0cmFfcmVwZWF0IjogYXJncy5leHRyYV9yZXBlYXQsCiAgICAgICAgInNwbGl0"
        "cyI6IHN0YXRzLAogICAgICAgICJjbGFzc19uYW1lcyI6IG1lcmdlZCwKICAgIH0KICAgIGlmIGFyZ3MucmVwb3J0OgogICAg"
        "ICAgIGFyZ3MucmVwb3J0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0xKSwgZW5jb2Rpbmc9InV0Zi04"
        "IikKICAgICAgICBwcmludChmIiAgcmVwb3J0IC0+IHthcmdzLnJlcG9ydH0iKQogICAgcmV0dXJuIHJlcG9ydAoKCmlmIF9f"
        "bmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"
    ),
}

TOOLS_DIR = pathlib.Path('/content/tools')
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
for name, blob in TOOLS_B64.items():
    (TOOLS_DIR / name).write_bytes(base64.b64decode(blob))
    print('wrote', TOOLS_DIR / name)

import sys
sys.path.insert(0, str(TOOLS_DIR))

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "/content/tools/repair_dataset.py",
     "--src", "/content/dataset",
     "--dst", "/content/dataset_repaired",
     "--report", "/content/repair_report.json"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("repair_dataset.py failed")

import json
rep = json.load(open("/content/repair_report.json"))
print("classes:", rep["classes_before"], "->", rep["classes_after"])
print("images :", rep["unique_images"],
      "(dropped", rep["dropped_duplicate_files"], "duplicate files)")
print("splits :", rep["splits"])
print("classes with <20 instances:", len(rep["rare_classes"]))

## Step 3b: Merge a Target-Style Dataset (optional)

If you have drawings in **your** drawing style, upload them here and
they will be merged into the training set with the class ids remapped by
name. This is the only thing that will teach the model your symbol
glyphs — see the note at the top of this notebook.

**Expected format** (a YOLO dataset, zipped):

```
target_style/
  data.yaml
  train/images/*.png
  train/labels/*.txt
  valid/...
  test/...
```

`data.yaml` must list the same class names the repaired dataset uses
(`rep["class_map"]` below shows the mapping). A ready-made starter set
built from `data/sample_pids/plant-pid.png` is described in
`data/target_style/glyph_catalogue.yaml` — the verified glyphs cut
out of `data/sample_pids/plant-pid.png`, which
`tools/augment_target.py` turns into new sheets in the same style with
exact ground truth.

**To skip this step, just run the next cell with `USE_TARGET_STYLE = False`.**

In [ ]:
USE_TARGET_STYLE = False   # set True after uploading the zip below
TARGET_ZIP = "/content/target_style.zip"

if USE_TARGET_STYLE:
    from google.colab import files
    print("Upload your target-style dataset zip:")
    uploaded = files.upload()
    if uploaded:
        next(iter(uploaded.values()))["metadata"].get("name", TARGET_ZIP)
        import shutil
        shutil.copy(next(iter(uploaded.values()))["name"], TARGET_ZIP)

if USE_TARGET_STYLE:
    import zipfile, pathlib
    if not pathlib.Path(TARGET_ZIP).exists():
        raise FileNotFoundError(f"{TARGET_ZIP} not found - upload it first")
    with zipfile.ZipFile(TARGET_ZIP) as z:
        z.extractall("/content/target_style_raw")
    # the zip may contain a single top-level folder
    roots = list(pathlib.Path("/content/target_style_raw").rglob("data.yaml"))
    if not roots:
        raise FileNotFoundError("no data.yaml inside the uploaded zip")
    target_root = roots[0].parent
    print("target dataset root:", target_root)

    result = subprocess.run(
        [sys.executable, "/content/tools/merge_datasets.py",
         "--base", "/content/dataset_repaired",
         "--extra", str(target_root),
         "--dst", "/content/dataset_merged",
         "--extra-repeat", "3",
         "--report", "/content/merge_report.json"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("merge_datasets.py failed")
    DATA_YAML = "/content/dataset_merged/data.yaml"
    DATASET_DIR = "/content/dataset_merged"
else:
    DATA_YAML = "/content/dataset_repaired/data.yaml"
    DATASET_DIR = "/content/dataset_repaired"

print("training on:", DATA_YAML)

## Step 4: Explore the Repaired Dataset

The previous version of this notebook printed the first 20 class names
and moved on. With 229 classes that hides everything, so this version
prints the **class health table**: how many instances each class has, and
which classes are too rare to learn.

In [ ]:
import glob, yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

data_config = yaml.safe_load(open(DATA_YAML))
names = data_config.get('names', [])
if isinstance(names, dict):
    names = [names[i] for i in sorted(names, key=int)]
print(f"classes: {len(names)}")
print(f"splits : { {k: v for k, v in data_config.items() if k in ('train','val','test')} }")

def count_instances(root):
    counter = Counter()
    for lbl in glob.glob(f'{root}/**/labels/*.txt', recursive=True):
        for line in open(lbl):
            p = line.split()
            if len(p) == 5:
                counter[int(p[0])] += 1
    return counter

train_counts = count_instances(data_config['train'])
counts = [train_counts.get(i, 0) for i in range(len(names))]

nonzero = [c for c in counts if c > 0]
print(f"\ninstances per class: min={min(nonzero)} median={sorted(nonzero)[len(nonzero)//2]} max={max(nonzero)}")
print(f"imbalance ratio: {max(nonzero)/max(1,min(nonzero)):.0f}x")
print(f"classes with <50 train instances: {sum(1 for c in counts if c < 50)}")

order = sorted(range(len(names)), key=lambda i: counts[i])
print("\n--- 20 rarest classes (annotate more of these) ---")
for i in order[:20]:
    print(f"  {counts[i]:6d}  {names[i]!r}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(counts)), sorted(counts))
ax.set_yscale('log')
ax.set_xlabel('class (sorted by frequency)')
ax.set_ylabel('training instances (log)')
ax.set_title('P&ID class frequency - the long tail is the training problem')
plt.tight_layout(); plt.show()

In [ ]:
# Visualise a few training images.
from PIL import Image
train_dir = Path(data_config['train'])
imgs = sorted(train_dir.glob('*.jpg'))[:6] or sorted(train_dir.glob('*.png'))[:6]
if imgs:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, p in zip(axes.flatten(), imgs):
        ax.imshow(Image.open(p).convert('RGB'))
        ax.set_title(p.name[:40], fontsize=8); ax.axis('off')
    plt.tight_layout(); plt.show()

## Step 5: Train

### Hyperparameters, and why

| Setting | Value | Why |
|---|---|---|
| `imgsz` | **1024** | The dataset has **44 different source image sizes** (225×225 to 1920×1080). At 640 the 1312×1312 sheets are downscaled 0.49× and small symbols vanish. |
| `epochs` | **300** | 229 classes with a 30-instance median needs a long schedule. The old `epochs=100, patience=15` stopped on the long tail. |
| `patience` | **50** | 15 epochs is shorter than the plateau on a 229-class problem. |
| `close_mosaic` | **15** | Mosaic blurs the fine line work of a symbol. Turning it off for the last 15 epochs is the cheapest accuracy win. |
| `mosaic` | 0.5 | Kept, but closed at the end. |
| `scale` | 0.2 | P&ID symbols are scale-sensitive; ±30% is too much. |
| `hsv_*` | 0.01 / 0.1 / 0.2 | Drawings are near-monochrome. |
| `fliplr` / `flipud` | 0.0 | Symbols have an orientation. |
| `degrees` | 5 | Scans are not perfectly level. |
| `erasing` | **0.0** | The default 0.4 *erases* 4% of every image — which deletes symbols. |
| `copy_paste` | 0.0 | Puts symbols on top of each other; not a real P&ID. |
| `auto_augment` | **None** | randaugment is for natural images. |
| `seed` | 42 | Reproducibility. |
| `optimizer` | `auto` | YOLOv8 picks AdamW for small datasets. |

In [ ]:
from ultralytics import YOLO

MODEL = 'yolov8s.pt'          # small: CPU inference on a laptop
EPOCHS = 300
PATIENCE = 50
IMGSZ = 1024
BATCH = 8
SEED = 42

model = YOLO(MODEL)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project='/content/runs',
    name='pid_yolov8s',
    exist_ok=True,

    # ---- schedule ----
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    workers=2,
    cache=False,
    save=True,
    save_period=25,
    plots=True,

    # ---- optimisation ----
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    amp=True,

    # ---- augmentation: gentle, and symbol-safe ----
    hsv_h=0.01, hsv_s=0.1, hsv_v=0.2,
    degrees=5,
    translate=0.1,
    scale=0.2,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,          # symbols have orientation
    fliplr=0.0,
    mosaic=0.5,
    close_mosaic=15,     # fine line work needs the last epochs clean
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,         # the default 0.4 deletes symbols
    auto_augment=None,   # randaugment is for natural images
)

## Step 6: Evaluate - Per Class, Not Just the Average

`mAP50 = 0.849` sounds excellent. On the shipped model it was measured
over **only 153 of 229 classes**; 76 classes were never scored, and 498
of the test images were byte-identical to training images. The number
was flattering and misleading. This section reports what you can act on.

In [ ]:
import torch
from ultralytics.utils import checks

run_dir = Path(results.save_dir)
best = run_dir / 'weights' / 'best.pt'

# Attach the training record to the checkpoint. The shipped best.pt had
# NO args, NO epoch counter and NO metrics, so there was no way to tell
# how it had been trained.
ckpt = torch.load(str(best), map_location='cpu', weights_only=False)
ckpt['train_args'] = {k: v for k, v in vars(model.trainer.args).items()}
ckpt['epoch'] = model.trainer.epoch
ckpt['train_metrics'] = getattr(model.trainer, 'metrics', None)
ckpt['date'] = getattr(model.trainer, 'date', None)
ckpt['version'] = ultralytics.__version__
ckpt['class_health'] = {
    'instances_per_class': {names[i]: counts[i] for i in range(len(names))},
    'classes_under_50_instances': sum(1 for c in counts if c < 50),
}
torch.save(ckpt, str(best))
print("re-saved with args:", best, f"{best.stat().st_size/1024/1024:.1f} MB")

In [ ]:
metrics = model.val(
    data=DATA_YAML,
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
)

box = metrics.box
print('=' * 60)
print(f"mAP50      {float(box.map50):.4f}")
print(f"mAP50-95   {float(box.map):.4f}")
print(f"precision  {float(box.mp):.4f}")
print(f"recall     {float(box.mr):.4f}")
print('=' * 60)

In [ ]:
# Per-class breakdown - the part that tells you what to fix.
per_class = []
for j, ci in enumerate(box.ap_class_index):
    per_class.append({'id': int(ci), 'name': names[int(ci)], 'map50': float(box.p[j])})
per_class.sort(key=lambda r: r['map50'])

evaluated = {r['id'] for r in per_class}
never = [names[i] for i in range(len(names)) if i not in evaluated]
zero = [r for r in per_class if r['map50'] == 0.0]

print(f"classes in vocabulary : {len(names)}")
print(f"classes EVALUATED      : {len(per_class)} ({100*len(per_class)/len(names):.1f}%)")
print(f"classes never evaluated: {len(never)}")
print(f"classes with map50 == 0: {len(zero)}")

print("\n--- 25 WORST classes ---")
for r in per_class[:25]:
    print(f"  {r['map50']:.3f}  {r['name']!r}")

print("\n--- 10 BEST classes ---")
for r in per_class[-10:][::-1]:
    print(f"  {r['map50']:.3f}  {r['name']!r}")

if never:
    print("\n--- classes never evaluated (missing from the test split) ---")
    for n in never[:40]:
        print(f"  {n!r}")

# A class that is rare AND scores zero is an annotation problem, not a
# model problem. This list is your annotation worklist.
print("\n--- zero-mAP classes ranked by training instances (annotate these) ---")
ranked = sorted((r for r in per_class if r['map50'] == 0.0),
                key=lambda r: counts[r['id']])
for r in ranked[:25]:
    print(f"  map50=0.000  train={counts[r['id']]:6d}  {r['name']!r}")

In [ ]:
from IPython.display import Image, display
import os

for fn in ('results.png', 'confusion_matrix.png', 'PR_curve.png'):
    p = run_dir / fn
    if os.path.exists(str(p)):
        display(Image(filename=str(p), width=900))

## Step 7: Export & Download the Trained Weights

In [ ]:
import os
from google.colab import files
from ultralytics import YOLO

best = run_dir / 'weights' / 'best.pt'
print(f"Best weights: {best}")
print(f"File size:    {os.path.getsize(best)/1024/1024:.1f} MB")

# Sanity check before it leaves Colab.
check = YOLO(str(best))
print("classes:", len(check.names))
assert len(check.names) == len(names), (
    f"class count mismatch: model has {len(check.names)}, dataset has {len(names)}"
)
print("OK - class count matches the dataset.")

files.download(str(best))
try:
    files.download('/content/repair_report.json')
except Exception:
    pass

## 📋 Next Steps

1. Download `best.pt`.
2. Copy it to `models/best.pt` in your project.
3. Run the pipeline: `streamlit run app/streamlit_app.py`

### Verify the new model against the old one

```bash
python tools/evaluate_model.py --model models/best.pt --data <dataset_dir> --split test
```

Compare `classes_evaluated` (must be the full vocabulary, not 153) and
`classes_with_zero_map`.

> **Note:** the repaired dataset has **218** classes, not 229 — the
> duplicate spellings were merged. The new checkpoint therefore has 218
> classes, and the old 229-class checkpoint is **not** interchangeable
> with it. `tools/evaluate_model.py` refuses to evaluate a model whose
> class list does not match the dataset's, so a mismatch cannot produce
> a plausible-looking but meaningless mAP.

### If the per-class report still shows zeros

The cause is almost always **annotation count**, not the model. A class
with 3 training instances cannot reach a useful mAP. Feed the
`-- 25 WORST classes --` list back to whoever annotated the drawings and
get each class to at least ~50 instances.

### Dataset Sources
- **Roboflow:** https://universe.roboflow.com/ai-ready-program/p-id-object-detection-ai-ready-pomcl
- **Hugging Face:** https://huggingface.co/datasets/hamzas/digitize-pid-yolo
- **Paper:** Paliwal et al. (2021) "Digitize-PID: Automatic Digitization of P&IDs" (arXiv:2109.03794)